# "A note on Cross Entropy loss with PyTorch"

- toc: true
- badges: true
- comments: true
- categories: [fastai, pytorch, jupyter]

In [1]:
from fastai.vision.all import *

Pretend the following are the activations of each class of a multiclass classification problem. So we have 6 examples and in each row we have the activation for each class the example could belong to.

In [2]:
activations = torch.randn((6,2))*2
activations

tensor([[ 2.0697, -2.1599],
        [ 0.4067,  0.6020],
        [-0.1373, -0.2908],
        [-2.3268,  2.9646],
        [-3.6268, -2.0845],
        [ 1.8271,  1.1254]])

Suppose the correct class of each example is as follows

In [3]:
targets = tensor([0,1,0,1,1,0])
targets

tensor([0, 1, 0, 1, 1, 0])

Take the softmax of the activations

In [4]:
sm_acts = torch.softmax(activations, dim=1)
sm_acts

tensor([[0.9857, 0.0143],
        [0.4513, 0.5487],
        [0.5383, 0.4617],
        [0.0050, 0.9950],
        [0.1762, 0.8238],
        [0.6686, 0.3314]])

Extract the probabilities predicted for the correct class.

In [5]:
idx = range(6)
list(idx)

[0, 1, 2, 3, 4, 5]

In [6]:
p_correct_class = sm_acts[idx, targets]
p_correct_class

tensor([0.9857, 0.5487, 0.5383, 0.9950, 0.8238, 0.6686])

Take the log of the softmax activations

In [7]:
torch.log(sm_acts)

tensor([[-1.4453e-02, -4.2441e+00],
        [-7.9555e-01, -6.0026e-01],
        [-6.1936e-01, -7.7282e-01],
        [-5.2965e+00, -5.0218e-03],
        [-1.7361e+00, -1.9383e-01],
        [-4.0263e-01, -1.1043e+00]])

Computing the softmax of the activations and then taking the log is equivalent to applying PyTorch's log_softmax function directly to the original activations. We want to do the latter because it will faster and more accurate.

In [8]:
torch.log_softmax(activations, dim=1)

tensor([[-1.4453e-02, -4.2441e+00],
        [-7.9555e-01, -6.0026e-01],
        [-6.1936e-01, -7.7282e-01],
        [-5.2965e+00, -5.0218e-03],
        [-1.7361e+00, -1.9383e-01],
        [-4.0263e-01, -1.1043e+00]])

Suppose we have $N$ training examples and we have a multiclass problem with $K$ classes. Let $C(i) \in \{1,\ldots, K\}$ be the correct label for the $i$-th training example and $p^{[C(i)]}_{i}$ gives the probability of the correct class for the $i$-th training example. We want to maximize: $$\prod_{i=1}^{N} p^{[C(i)]}_{i}$$ 

This is equivalent to maximizing $$ln(\prod_{i=1}^{N}p^{[C(i)]}_{i}) = \sum_{i=1}^{N}ln(p^{[C(i)]}_{i})$$ 

Which is the same as minimizing the sum of the negative log likelihoods $$-\sum_{i=1}^{N}ln(p^{[C(i)]}_{i})$$

The above can now serve as a loss function.

Recall Cross Entropy = $-\sum_{k=1}^{K}q^{[k]}ln(p^{[k]})$ where $q$ is the reference distribution over $K$ classes while our predictions over the $K$ classes have the distribution $p$. Observe this becomes just a single term when, in the reference distribution $q$, only one of the classes has a probability of $1$.

Thus, $-\sum_{i=1}^{N}ln(p^{[C(i)]}_{i})$ can be interpreted as the sum of cross entropy losses across all examples.

Next we can compute the above quantity as follows:

In [9]:
-1*torch.log(p_correct_class), (-1*torch.log(p_correct_class)).mean()

(tensor([0.0145, 0.6003, 0.6194, 0.0050, 0.1938, 0.4026]), tensor(0.3059))

We can use Pytorch to compute this directly

In [10]:
nn.CrossEntropyLoss(reduction='none')(activations, targets), nn.CrossEntropyLoss()(activations, targets)

(tensor([0.0145, 0.6003, 0.6194, 0.0050, 0.1938, 0.4026]), tensor(0.3059))

or by using:

In [11]:
F.cross_entropy(activations, targets, reduction='none'), F.cross_entropy(activations, targets)

(tensor([0.0145, 0.6003, 0.6194, 0.0050, 0.1938, 0.4026]), tensor(0.3059))

# Gradient of Cross Entropy

We follow the note in {% cite markusthill_ce_note %}

# References
{% bibliography --cited %}